# Case Study 3: E-Commerce Product Search & Recommendation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TechyNilesh/DeepTextSearch/blob/main/notebooks/03_case_study_ecommerce_product_search.ipynb)

**Goal**: Build a product search and recommendation system for an e-commerce store with:
- CSV data loading with rich metadata
- Multi-field metadata filtering (price, brand, category)
- Product recommendation ("similar to this product")
- Search weight tuning for different use cases

**Features covered**: `from_csv()`, DataFrame indexing, metadata columns, `filter_fn` with complex conditions, dense search for recommendations, `rerank_texts()`, `SearchResult.to_dict()`, hybrid weight tuning

In [1]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
!pip install -q git+https://github.com/TechyNilesh/DeepTextSearch.git

## Step 1: Create Product Catalog

In [3]:
import pandas as pd
import tempfile, os

products = pd.DataFrame({
    "name": [
        "MacBook Pro 16-inch M3 Max",
        "Dell XPS 15 OLED",
        "ThinkPad X1 Carbon Gen 11",
        "Sony WH-1000XM5 Headphones",
        "AirPods Pro 2nd Gen",
        "Bose QuietComfort Ultra",
        "Samsung Galaxy S24 Ultra",
        "iPhone 15 Pro Max",
        "Google Pixel 8 Pro",
        "iPad Pro 12.9-inch M2",
        "Samsung Galaxy Tab S9 Ultra",
        "Kindle Paperwhite Signature",
        "LG C3 65-inch OLED TV",
        "Sony A95L QD-OLED TV",
        "Logitech MX Master 3S Mouse",
        "Apple Magic Keyboard",
        "Corsair K100 Mechanical Keyboard",
        "Samsung 990 Pro 2TB SSD",
        "WD Black SN850X NVMe SSD",
        "ASUS ROG Strix RTX 4090 GPU",
    ],
    "description": [
        "The most powerful MacBook ever with M3 Max chip, 36GB unified memory, stunning Liquid Retina XDR display, and up to 22 hours of battery life for professional workflows.",
        "Premium ultrabook with a gorgeous 15.6-inch 3.5K OLED display, 13th Gen Intel Core i7, 16GB RAM, and a sleek aluminum chassis weighing just 4.23 lbs.",
        "Ultra-lightweight business laptop at just 2.48 lbs with 13th Gen Intel vPro, 14-inch 2.8K OLED display, and legendary ThinkPad keyboard for all-day productivity.",
        "Industry-leading noise cancellation headphones with 30-hour battery life, multipoint connection, adaptive sound control, and premium audio quality.",
        "Compact wireless earbuds with active noise cancellation, adaptive transparency mode, personalized spatial audio, and MagSafe charging case.",
        "Premium over-ear headphones with immersive spatial audio, world-class noise cancellation, and luxurious comfort for extended listening sessions.",
        "Flagship smartphone with 200MP camera, S Pen support, titanium frame, 6.8-inch Dynamic AMOLED display, and powerful Snapdragon 8 Gen 3 processor.",
        "Apple's most advanced iPhone with A17 Pro chip, titanium design, 48MP camera system with 5x optical zoom, and USB-C with USB 3 speeds.",
        "Google's AI-first smartphone with Tensor G3 chip, best-in-class computational photography, 7 years of software updates, and advanced AI features.",
        "Professional tablet with M2 chip, 12.9-inch Liquid Retina XDR display, Thunderbolt connectivity, and Apple Pencil hover support for creative work.",
        "Massive 14.6-inch Super AMOLED tablet with Snapdragon 8 Gen 2, S Pen included, DeX desktop mode, and IP68 water resistance.",
        "Premium e-reader with 6.8-inch display, wireless charging, adjustable warm light, 32GB storage, and auto-adjusting front light for comfortable reading.",
        "Stunning 65-inch OLED TV with 4K resolution, 120Hz refresh rate, Dolby Vision, webOS smart platform, and ultra-thin gallery design.",
        "Premium 65-inch QD-OLED TV with unmatched color accuracy, Cognitive Processor XR, HDMI 2.1, and Google TV built-in for the ultimate viewing experience.",
        "Ergonomic wireless mouse with MagSpeed scroll wheel, 8K DPI sensor, USB-C charging, and seamless multi-device workflow across three computers.",
        "Slim wireless keyboard with Touch ID, scissor mechanism keys, USB-C charging, and seamless integration with Mac and iPad devices.",
        "High-performance mechanical gaming keyboard with optical-mechanical switches, 44-zone RGB lighting, iCUE software integration, and detachable wrist rest.",
        "Ultra-fast PCIe Gen 4 NVMe SSD with sequential read speeds up to 7,450 MB/s, hardware encryption, and Samsung V-NAND technology.",
        "High-performance NVMe SSD with up to 7,300 MB/s read speeds, Game Mode 2.0, and optimized for gaming and creative professional workloads.",
        "Ultimate gaming GPU with 24GB GDDR6X memory, record-breaking performance in 4K gaming, ray tracing, and AI-powered DLSS 3 frame generation.",
    ],
    "category": [
        "Laptops", "Laptops", "Laptops",
        "Audio", "Audio", "Audio",
        "Phones", "Phones", "Phones",
        "Tablets", "Tablets", "E-Readers",
        "TVs", "TVs",
        "Peripherals", "Peripherals", "Peripherals",
        "Storage", "Storage", "GPUs",
    ],
    "brand": [
        "Apple", "Dell", "Lenovo",
        "Sony", "Apple", "Bose",
        "Samsung", "Apple", "Google",
        "Apple", "Samsung", "Amazon",
        "LG", "Sony",
        "Logitech", "Apple", "Corsair",
        "Samsung", "WD", "ASUS",
    ],
    "price": [
        3499, 1899, 1649,
        349, 249, 429,
        1299, 1199, 899,
        1099, 1199, 189,
        1799, 2799,
        99, 199, 229,
        179, 149, 1999,
    ],
})

# Save as CSV for demo
csv_path = os.path.join(tempfile.gettempdir(), "products.csv")
products.to_csv(csv_path, index=False)

print(f"Product catalog: {len(products)} items")
print(f"Categories: {products['category'].nunique()} — {sorted(products['category'].unique())}")
print(f"Price range: ${products['price'].min()} — ${products['price'].max()}")
products[["name", "category", "brand", "price"]].head(10)

Product catalog: 20 items
Categories: 9 — ['Audio', 'E-Readers', 'GPUs', 'Laptops', 'Peripherals', 'Phones', 'Storage', 'TVs', 'Tablets']
Price range: $99 — $3499


,name,category,brand,price
0,MacBook Pro 16-inch M3 Max,Laptops,Apple,3499
1,Dell XPS 15 OLED,Laptops,Dell,1899
2,ThinkPad X1 Carbon Gen 11,Laptops,Lenovo,1649
3,Sony WH-1000XM5 Headphones,Audio,Sony,349
4,AirPods Pro 2nd Gen,Audio,Apple,249
5,Bose QuietComfort Ultra,Audio,Bose,429
6,Samsung Galaxy S24 Ultra,Phones,Samsung,1299
7,iPhone 15 Pro Max,Phones,Apple,1199
8,Google Pixel 8 Pro,Phones,Google,899
9,iPad Pro 12.9-inch M2,Tablets,Apple,1099


## Step 2: Index from CSV

In [4]:
from DeepTextSearch import TextEmbedder, TextSearch, Reranker

# One-liner: load CSV and index
embedder = TextEmbedder.from_csv(
    csv_path,
    text_column="description",
    metadata_columns=["name", "category", "brand", "price"],
    model_name="BAAI/bge-small-en-v1.5",
)

search = TextSearch(embedder, mode="hybrid")
print(f"Indexed {embedder.corpus_size} products")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Indexed 20 products


## Step 3: Product Search

In [5]:
def display_results(results, show_price=True):
    for i, r in enumerate(results, 1):
        price = f"${r.metadata.get('price', '?')}" if show_price else ""
        print(f"  {i}. [{r.score:.6f}] {r.metadata['name']} {price}")
        print(f"     {r.metadata.get('brand', '')} | {r.metadata.get('category', '')}")
        print(f"     {r.text[:80]}...\n")

# Natural language product search
print("=== 'best laptop for programming' ===")
display_results(search.search("best laptop for programming and development", top_n=3))

print("=== 'wireless noise cancelling headphones' ===")
display_results(search.search("wireless noise cancelling headphones for travel", top_n=3))

print("=== 'fast storage for gaming' ===")
display_results(search.search("fast SSD storage for gaming PC build", top_n=3))

=== 'best laptop for programming' ===
  1. [0.016393] ThinkPad X1 Carbon Gen 11 $1649
     Lenovo | Laptops
     Ultra-lightweight business laptop at just 2.48 lbs with 13th Gen Intel vPro, 14-...

  2. [0.015385] iPad Pro 12.9-inch M2 $1099
     Apple | Tablets
     Professional tablet with M2 chip, 12.9-inch Liquid Retina XDR display, Thunderbo...

  3. [0.015345] MacBook Pro 16-inch M3 Max $3499
     Apple | Laptops
     The most powerful MacBook ever with M3 Max chip, 36GB unified memory, stunning L...

=== 'wireless noise cancelling headphones' ===
  1. [0.016185] AirPods Pro 2nd Gen $249
     Apple | Audio
     Compact wireless earbuds with active noise cancellation, adaptive transparency m...

  2. [0.016129] Sony WH-1000XM5 Headphones $349
     Sony | Audio
     Industry-leading noise cancellation headphones with 30-hour battery life, multip...

  3. [0.016081] Bose QuietComfort Ultra $429
     Bose | Audio
     Premium over-ear headphones with immersive spatial audio, world-cl

## Step 4: Filtered Search — By Category, Brand, and Price

In [6]:
# Apple products only
print("=== Apple Products Only ===")
results = search.search(
    "portable device for creative work",
    top_n=5,
    filter_fn=lambda text, meta: meta.get("brand") == "Apple",
)
display_results(results)

# Budget products under $300
print("=== Budget (Under $300) ===")
results = search.search(
    "best audio experience",
    top_n=5,
    filter_fn=lambda text, meta: meta.get("price", 9999) < 300,
)
display_results(results)

# Samsung phones and tablets
print("=== Samsung Phones & Tablets ===")
results = search.search(
    "large display mobile device",
    top_n=5,
    filter_fn=lambda text, meta: (
        meta.get("brand") == "Samsung"
        and meta.get("category") in ["Phones", "Tablets"]
    ),
)
display_results(results)

=== Apple Products Only ===
  1. [0.016393] iPad Pro 12.9-inch M2 $1099
     Apple | Tablets
     Professional tablet with M2 chip, 12.9-inch Liquid Retina XDR display, Thunderbo...

  2. [0.014421] MacBook Pro 16-inch M3 Max $3499
     Apple | Laptops
     The most powerful MacBook ever with M3 Max chip, 36GB unified memory, stunning L...

  3. [0.009524] Apple Magic Keyboard $199
     Apple | Peripherals
     Slim wireless keyboard with Touch ID, scissor mechanism keys, USB-C charging, an...

  4. [0.009375] AirPods Pro 2nd Gen $249
     Apple | Audio
     Compact wireless earbuds with active noise cancellation, adaptive transparency m...

  5. [0.008696] iPhone 15 Pro Max $1199
     Apple | Phones
     Apple's most advanced iPhone with A17 Pro chip, titanium design, 48MP camera sys...

=== Budget (Under $300) ===
  1. [0.009524] AirPods Pro 2nd Gen $249
     Apple | Audio
     Compact wireless earbuds with active noise cancellation, adaptive transparency m...

  2. [0.009375] Corsai

## Step 5: Product Recommendations ("Similar to this")

Use dense search to find semantically similar products.

In [7]:
# User is viewing the MacBook Pro — recommend similar products
source_product = products.iloc[0]
print(f"Viewing: {source_product['name']} (${source_product['price']})")
print(f"→ Finding similar products...\n")

# Use the product description as the query with dense search
similar = search.search(
    source_product["description"],
    top_n=5,
    mode="dense",  # Pure semantic similarity
)

# Skip the first result (it's the same product)
print("=== Similar Products ===")
for i, r in enumerate(similar[1:], 1):
    print(f"  {i}. {r.metadata['name']} — ${r.metadata['price']} ({r.metadata['category']})")
    print(f"     Similarity: {r.score:.4f}")

Viewing: MacBook Pro 16-inch M3 Max ($3499)
→ Finding similar products...

=== Similar Products ===
  1. Dell XPS 15 OLED — $1899 (Laptops)
     Similarity: 0.7325
  2. iPad Pro 12.9-inch M2 — $1099 (Tablets)
     Similarity: 0.6775
  3. iPhone 15 Pro Max — $1199 (Phones)
     Similarity: 0.6663
  4. ThinkPad X1 Carbon Gen 11 — $1649 (Laptops)
     Similarity: 0.6387


## Step 6: Rerank Product Results

In [8]:
reranker = Reranker("cross-encoder/ms-marco-MiniLM-L-6-v2")

query = "best smartphone with amazing camera for photography"
candidates = search.search(query, top_n=10)

print(f"Query: '{query}'\n")
print("=== Before Reranking ===")
for i, r in enumerate(candidates[:5], 1):
    print(f"  {i}. {r.metadata['name']}")

reranked = reranker.rerank_search_results(query, candidates, top_n=5)
print("\n=== After Reranking ===")
for i, r in enumerate(reranked, 1):
    print(f"  {i}. [{r['score']:.4f}] {r['metadata']['name']}")

Query: 'best smartphone with amazing camera for photography'

=== Before Reranking ===
  1. iPhone 15 Pro Max
  2. Samsung Galaxy S24 Ultra
  3. Google Pixel 8 Pro
  4. iPad Pro 12.9-inch M2
  5. MacBook Pro 16-inch M3 Max

=== After Reranking ===
  1. [3.3276] Google Pixel 8 Pro
  2. [1.9821] Samsung Galaxy S24 Ultra
  3. [-0.0240] iPhone 15 Pro Max
  4. [-7.9487] Samsung Galaxy Tab S9 Ultra
  5. [-9.0847] Kindle Paperwhite Signature


## Step 7: Search Result as Dict (API Response)

In [9]:
import json

results = search.search("gaming keyboard RGB", top_n=3)

# Convert to list of dicts — ready for JSON API response
api_response = [r.to_dict() for r in results]

print("=== API Response (JSON) ===")
print(json.dumps(api_response, indent=2))

=== API Response (JSON) ===
[
  {
    "index": 16,
    "text": "High-performance mechanical gaming keyboard with optical-mechanical switches, 44-zone RGB lighting, iCUE software integration, and detachable wrist rest.",
    "score": 0.016393,
    "metadata": {
      "name": "Corsair K100 Mechanical Keyboard",
      "category": "Peripherals",
      "brand": "Corsair",
      "price": 229
    }
  },
  {
    "index": 15,
    "text": "Slim wireless keyboard with Touch ID, scissor mechanism keys, USB-C charging, and seamless integration with Mac and iPad devices.",
    "score": 0.016129,
    "metadata": {
      "name": "Apple Magic Keyboard",
      "category": "Peripherals",
      "brand": "Apple",
      "price": 199
    }
  },
  {
    "index": 19,
    "text": "Ultimate gaming GPU with 24GB GDDR6X memory, record-breaking performance in 4K gaming, ray tracing, and AI-powered DLSS 3 frame generation.",
    "score": 0.015873,
    "metadata": {
      "name": "ASUS ROG Strix RTX 4090 GPU",
      

## Step 8: Tuning Hybrid Weights

Compare keyword-heavy vs semantic-heavy search.

In [10]:
query = "OLED display"

# Keyword-heavy (favors exact term matches)
search_kw = TextSearch(embedder, dense_weight=0.2, bm25_weight=0.8)
print("=== Keyword-Heavy (bm25=0.8) ===")
for r in search_kw.search(query, top_n=3):
    print(f"  {r.metadata['name']}")

# Semantic-heavy (favors conceptual meaning)
search_sem = TextSearch(embedder, dense_weight=0.8, bm25_weight=0.2)
print("\n=== Semantic-Heavy (dense=0.8) ===")
for r in search_sem.search(query, top_n=3):
    print(f"  {r.metadata['name']}")

# Balanced (default)
print("\n=== Balanced (default) ===")
for r in search.search(query, top_n=3):
    print(f"  {r.metadata['name']}")

=== Keyword-Heavy (bm25=0.8) ===
  LG C3 65-inch OLED TV
  Dell XPS 15 OLED
  ThinkPad X1 Carbon Gen 11

=== Semantic-Heavy (dense=0.8) ===
  LG C3 65-inch OLED TV
  Dell XPS 15 OLED
  Sony A95L QD-OLED TV

=== Balanced (default) ===
  LG C3 65-inch OLED TV
  Dell XPS 15 OLED
  Sony A95L QD-OLED TV


## Step 9: Save Index for Production

In [11]:
# Save
embedder.save("./product_search_index")
print(f"Saved! Files: {os.listdir('./product_search_index')}")

# Verify it loads correctly
loaded = TextEmbedder.load("./product_search_index")
search_loaded = TextSearch(loaded)
print(f"Loaded: {loaded.corpus_size} products")

# Quick search on loaded index
for r in search_loaded.search("4K OLED TV", top_n=2):
    print(f"  {r.metadata['name']} — ${r.metadata['price']}")

Saved! Files: ['index.faiss', 'store_meta.json', 'corpus.json', 'config.json']


Loaded: 20 products
  LG C3 65-inch OLED TV — $1799
  Sony A95L QD-OLED TV — $2799


---

**Summary**: This case study built a complete e-commerce product search engine with CSV loading, rich metadata filtering (brand, category, price range), product recommendations via dense similarity, reranking for precision, hybrid weight tuning, and JSON API output.